# Step 3: Projection Model

Builds the per-player-per-week fantasy point projection model. Built up piece by piece, matching `ROADMAP.md` Step 3:
1. Multi-season raw data pull
2. Feature engineering (trailing form, season-to-date/prior-season, categorical)
3. Train/val/test split by season
4. Baseline model
5. LightGBM model
6. Evaluation (per position)


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from nflverse_loader import load_players, load_weekly_stats
from data_loader import normalize_player_week


## 1. Multi-season raw data pull

2019-2024: recent enough that the modern passing-heavy game context still applies, enough volume for weekly-level training. Train 2019-2022, validate 2023, test 2024 (picked in piece 6).

Two real bugs surfaced and fixed while wiring this up (see `PROJECTS.md`/`ROADMAP.md` for detail): `PLAYERS_SCHEMA` was missing `season` (team/bye_week are season-dependent, not identity metadata — only worked before because every caller pulled one season at a time), and 2022's bye-week derivation double-counted the Week 17 Bills-Bengals game (suspended after Damar Hamlin's on-field cardiac arrest, never replayed) as a phantom bye for both teams.

In [2]:
SEASONS = list(range(2019, 2025))

players = load_players(SEASONS)
weekly = load_weekly_stats(SEASONS)
player_week = normalize_player_week(players, weekly)

print(f"players: {players.shape}, weekly: {weekly.shape}, player_week: {player_week.shape}")
player_week["season"].value_counts().sort_index()


players: (18576, 6), weekly: (39224, 14), player_week: (39224, 19)


season
2019    6165
2020    6354
2021    6702
2022    6653
2023    6640
2024    6710
Name: count, dtype: int64

In [3]:
# Sanity check: fantasy points by position, roughly matches expected scale
# (QB highest, then RB/WR, TE a bit lower, K always 0 — kicker scoring isn't
# wired into WEEKLY_STATS_SCHEMA yet, see schema.py).
player_week.groupby("position")["fantasy_points"].agg(["count", "mean"]).round(2)


,count,mean
position,,
DB,29,0.51
DL,1,0.00
K,3330,0.00
LB,9,0.11
P,15,0.00
QB,4065,14.02
RB,9397,8.00
TE,7280,5.63
WR,15098,7.61


## 2. Feature engineering: trailing form

Rolling averages of a player's own `fantasy_points`, computed causally (shifted so a given week's features never see that week's own result). Carries across season boundaries rather than resetting to NaN each September - a player's last 3 games are still the most relevant signal of current form.


In [4]:
from features import add_trailing_form_features

featured = add_trailing_form_features(player_week)
featured[["trailing_3g_avg", "trailing_5g_avg"]].describe()


,trailing_3g_avg,trailing_5g_avg
count,37916.000000,37916.000000
mean,7.468254,7.483758
std,6.749042,6.421094
min,-2.780000,-2.780000
25%,1.766667,2.120000
50%,5.833333,6.000000
75%,11.733333,11.740000
max,46.800000,46.800000


In [5]:
# Eyeball check against a real player's game log, including a season boundary
# and a real injury gap (McCaffrey missed 2024 weeks 1-9).
sample_id = "00-0033280"  # Christian McCaffrey
cols = ["name", "season", "week", "fantasy_points", "trailing_3g_avg", "trailing_5g_avg"]
featured.loc[featured["player_id"] == sample_id, cols].sort_values(["season", "week"]).tail(12)


,name,season,week,fantasy_points,trailing_3g_avg,trailing_5g_avg
30121,Christian McCaffrey,2023,13,22.3,24.133333,24.96
30435,Christian McCaffrey,2023,14,16.3,24.833333,24.90
30795,Christian McCaffrey,2023,15,41.7,23.166667,22.20
31180,Christian McCaffrey,2023,16,25.1,26.766667,26.50
31547,Christian McCaffrey,2023,17,13.1,27.700000,27.26
32367,Christian McCaffrey,2023,20,31.8,26.633333,23.70
32452,Christian McCaffrey,2023,21,29.2,23.333333,25.60
32493,Christian McCaffrey,2023,22,28.0,24.700000,28.18
35791,Christian McCaffrey,2024,10,16.7,29.666667,25.44
36121,Christian McCaffrey,2024,11,14.6,24.633333,23.76


## 3. Feature engineering: season-to-date + prior-season

`season_to_date_avg` deliberately resets each season - a different question ("how is this player doing this year") than trailing form. `prior_season_*` covers the cold-start gap early in a season: a returning veteran's last full season is a much better prior than nothing for Week 1-3. True rookies still end up NaN (no prior season in this dataset at all) - left for LightGBM to handle natively.


In [6]:
from features import add_season_to_date_features, add_prior_season_features

featured = add_season_to_date_features(featured)
featured = add_prior_season_features(featured)
featured[["season_to_date_avg", "prior_season_total_points", "prior_season_avg_points"]].describe()


,season_to_date_avg,prior_season_total_points,prior_season_avg_points
count,35379.000000,26941.000000,26941.000000
mean,7.417576,110.484619,7.781264
std,6.456293,103.405082,6.105792
min,-2.860000,-2.880000,-2.880000
25%,1.950000,21.100000,2.633333
50%,5.975000,85.200000,6.807143
75%,11.746154,174.700000,12.142857
max,46.800000,554.800000,30.133333


In [7]:
# Verify against McCaffrey's real 2022 -> 2023 transition.
cols = ["name", "season", "week", "fantasy_points", "season_to_date_avg",
        "prior_season_total_points", "prior_season_avg_points"]
featured.loc[(featured["player_id"] == "00-0033280") & (featured["season"] == 2023), cols].head(5)


,name,season,week,fantasy_points,season_to_date_avg,prior_season_total_points,prior_season_avg_points
11804,Christian McCaffrey,2023,1,25.9,NaN,416.26,20.813
11805,Christian McCaffrey,2023,2,22.5,25.900000,416.26,20.813
11806,Christian McCaffrey,2023,3,22.9,24.200000,416.26,20.813
11807,Christian McCaffrey,2023,4,48.7,23.766667,416.26,20.813
11808,Christian McCaffrey,2023,5,13.8,30.000000,416.26,20.813


## 4. Feature engineering: categorical/context

Position and team, cast to pandas `category` dtype so LightGBM can split on them natively (no one-hot encoding). A "bye-week flag" was in the original plan for this piece but got dropped: every row here already represents a game the player actually played (byes produce no stat line at all), so the flag could never be True on any real row.

`build_features()` now wraps all four pieces (trailing form, season-to-date, prior-season, categorical) into one call - used from here on instead of chaining the individual functions.


In [8]:
from features import build_features, FEATURE_COLUMNS

featured = build_features(player_week)
featured[["position", "team"]].dtypes


position    category
team        category
dtype: object

In [9]:
FEATURE_COLUMNS


['trailing_3g_avg',
 'trailing_5g_avg',
 'season_to_date_avg',
 'prior_season_total_points',
 'prior_season_games_played',
 'prior_season_avg_points',
 'position',
 'team']

## 5. Train/val/test split

Scoped to QB/RB/WR/TE only: K's fantasy_points is a data gap pretending to be a real zero (kicker FG/PAT stats aren't wired into WEEKLY_STATS_SCHEMA yet), and the handful of DB/DL/LB/P rows are gadget-play noise, not standard fantasy production.

Split **by season**, not by row - train 2019-2022, validate 2023, test 2024 - so no future season ever leaks into training.


In [10]:
from projection_model import filter_to_model_positions, split_by_season

modelable = filter_to_model_positions(featured)
train, val, test = split_by_season(modelable, train_seasons=[2019, 2020, 2021, 2022], val_season=2023, test_season=2024)

print(f"train: {train.shape} ({sorted(train['season'].unique())})")
print(f"val:   {val.shape} ({sorted(val['season'].unique())})")
print(f"test:  {test.shape} ({sorted(test['season'].unique())})")


train: (23637, 25) ([np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)])
val:   (6065, 25) ([np.int64(2023)])
test:  (6138, 25) ([np.int64(2024)])


In [11]:
# Position mix should look similar across splits - no season should be skewing
# heavily toward one position (would suggest a filtering bug).
for name, split in [("train", train), ("val", val), ("test", test)]:
    print(name, dict(split["position"].value_counts()))


train {'WR': np.int64(9940), 'RB': np.int64(6262), 'TE': np.int64(4774), 'QB': np.int64(2661), 'DB': np.int64(0), 'DL': np.int64(0), 'K': np.int64(0), 'LB': np.int64(0), 'P': np.int64(0)}
val {'WR': np.int64(2599), 'RB': np.int64(1527), 'TE': np.int64(1232), 'QB': np.int64(707), 'DB': np.int64(0), 'DL': np.int64(0), 'K': np.int64(0), 'LB': np.int64(0), 'P': np.int64(0)}
test {'WR': np.int64(2559), 'RB': np.int64(1608), 'TE': np.int64(1274), 'QB': np.int64(697), 'DB': np.int64(0), 'DL': np.int64(0), 'K': np.int64(0), 'LB': np.int64(0), 'P': np.int64(0)}


## 6. Baseline model

The trailing-3-game average *is* the baseline prediction, with a cascading fallback for rows where it's NaN (a player's first tracked game, or first game of a season): trailing average -> prior-season average -> the position's overall training-set average (a true rookie's very first game has neither). The fallback average is fit from training data only. This is the bar LightGBM has to beat.


In [12]:
from sklearn.metrics import mean_absolute_error
from projection_model import fit_baseline, baseline_predict

position_fallback = fit_baseline(train)
val_baseline_pred = baseline_predict(val, position_fallback)

assert val_baseline_pred.isna().sum() == 0, "baseline should never be NaN after fallback"

print("Baseline MAE by position (validation, 2023):")
for pos in ["QB", "RB", "WR", "TE"]:
    mask = val["position"] == pos
    mae = mean_absolute_error(val.loc[mask, "fantasy_points"], val_baseline_pred[mask])
    print(f"  {pos}: {mae:.2f}")


Baseline MAE by position (validation, 2023):
  QB: 6.42
  RB: 4.73
  WR: 4.79
  TE: 3.82


## 7. LightGBM model

Early-stops against the validation set (2023) to pick the number of trees - the held-out test set (2024) stays completely untouched until the final evaluation in the next section.


In [13]:
from projection_model import train_lightgbm

model = train_lightgbm(train, val)
print(f"best iteration: {model.best_iteration_}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000972 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1336
[LightGBM] [Info] Number of data points in the train set: 23637, number of used features: 8
[LightGBM] [Info] Start training from score 8.138777
best iteration: 63


In [14]:
# Quick sanity check on validation (the set it was early-stopped against,
# not a clean holdout - the real comparison against baseline happens on
# the untouched test set next).
val_model_pred = model.predict(val[FEATURE_COLUMNS])
print("LightGBM MAE by position (validation, 2023):")
for pos in ["QB", "RB", "WR", "TE"]:
    mask = (val["position"] == pos).to_numpy()
    mae = mean_absolute_error(val.loc[mask, "fantasy_points"], val_model_pred[mask])
    print(f"  {pos}: {mae:.2f}")


LightGBM MAE by position (validation, 2023):
  QB: 6.07
  RB: 4.58
  WR: 4.62
  TE: 3.63
